In [ ]:
import cv2
import torch
import numpy as np

def read_video_for_tracker(mp4_path, max_frames=1000):
    cap = cv2.VideoCapture(mp4_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Unable to open video file: {mp4_path}")
    
    frames = []
    
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    
    if len(frames) == 0:
        raise ValueError("Video has 0 frames, please check if the file is corrupted")
    
    print(f"Read {len(frames)} frames, resolution: {frames[0].shape}")
    
    video_tensor = (
        torch.tensor(np.array(frames))
        .permute(0, 3, 1, 2)
        .unsqueeze(0)
        .float()
    )
    return video_tensor

In [ ]:
# Cell 2: Annotation script
import cv2
import json

video_path = "./dataset/dynamic_seq_003.mp4"
MAX_FRAMES = 30

cap = cv2.VideoCapture(video_path)
frames = []
while len(frames) < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()

first_frame = frames[0]
last_frame = frames[-1]
print(f"Read {len(frames)} frames in total, annotating frame 0 and frame {len(frames)-1}")

points = {"f0": [], "fLast": [], "total_frames": len(frames)}

def click_event(event, x, y, flags, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        params.append([x, y])
        print(f"Point recorded: ({x}, {y}), total {len(params)} points so far")

cv2.imshow("First Frame - click 5 points, press any key to confirm", first_frame)
cv2.setMouseCallback("First Frame - click 5 points, press any key to confirm", click_event, points["f0"])
cv2.waitKey(0)
print(f"First frame annotated with {len(points['f0'])} points")

cv2.imshow("Last Frame (frame 50) - click same 5 points, press any key to confirm", last_frame)
cv2.setMouseCallback("Last Frame (frame 50) - click same 5 points, press any key to confirm", click_event, points["fLast"])
cv2.waitKey(0)
print(f"Last frame annotated with {len(points['fLast'])} points")

cv2.destroyAllWindows()

assert len(points["f0"]) == len(points["fLast"]), \
    f"Number of points inconsistent: first frame {len(points['f0'])}, last frame {len(points['fLast'])}"

with open("annotation_003.json", "w") as f:
    json.dump(points, f, indent=2)
print("Annotations saved to annotation_003.json")

共读取 30 帧，标注第0帧和第29帧


2026-03-09 17:52:37.485 python[26790:8423171] error messaging the mach port for IMKCFRunLoopWakeUpReliable


已记录点: (232, 223)，当前共 1 个点
已记录点: (277, 194)，当前共 2 个点
已记录点: (230, 285)，当前共 3 个点
已记录点: (330, 224)，当前共 4 个点
已记录点: (312, 297)，当前共 5 个点
第一帧共标注 5 个点
已记录点: (201, 193)，当前共 1 个点
已记录点: (238, 126)，当前共 2 个点
已记录点: (219, 252)，当前共 3 个点
已记录点: (293, 118)，当前共 4 个点
已记录点: (321, 225)，当前共 5 个点
最后帧共标注 5 个点
标注已保存到 annotation_003.json


In [15]:
%cd EndoTrack/

[Errno 2] No such file or directory: 'EndoTrack/'
/Users/cx/Desktop/EndoTrack


In [16]:
%cd co-tracker/

/Users/cx/Desktop/EndoTrack/co-tracker


In [17]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
model = torch.hub.load("facebookresearch/co-tracker", "cotracker2").to(device)
model.eval()

使用设备: cpu


Using cache found in /Users/cx/.cache/torch/hub/facebookresearch_co-tracker_main


CoTrackerPredictor(
  (model): CoTracker2(
    (fnet): BasicEncoder(
      (norm1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (norm2): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
      (relu1): ReLU(inplace=True)
      (layer1): Sequential(
        (0): ResidualBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          (norm1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          (norm2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        )
        (1): ResidualBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Con

In [ ]:
# Cell 6: Tracking and evaluation
import json
import torch
import numpy as np
from cotracker.utils.visualizer import Visualizer
import os, cv2

%cd ..
with open("annotation_003.json", "r") as f:
    ann = json.load(f)

f0_pts = ann["f0"]
fL_pts = ann["fLast"]

queries = torch.tensor(
    [[0.0, x, y] for x, y in f0_pts]
).unsqueeze(0).to(device)

video_tensor = read_video_for_tracker(video_path, max_frames=MAX_FRAMES).to(device)
T = video_tensor.shape[1]
H = video_tensor.shape[3]
W = video_tensor.shape[4] if video_tensor.dim() == 5 else video_tensor.shape[3]
H, W = video_tensor.shape[3], video_tensor.shape[4]

torch.cuda.empty_cache()
with torch.no_grad():
    pred_tracks, pred_vis = model(video_tensor, queries=queries)

def tapvid_delta_avg(pred_last, gt_last, pred_vis_last, video_H, video_W):
    diag = np.sqrt(video_H**2 + video_W**2)
    thresholds = np.array([1, 2, 4, 8, 16]) / 256.0 * diag

    visible_mask = pred_vis_last > 0.5
    if visible_mask.sum() == 0:
        print(" All points are invisible, cannot evaluate")
        return None

    pred_vis_pts = pred_last[visible_mask]
    gt_vis_pts = gt_last[visible_mask]

    errors = np.linalg.norm(pred_vis_pts - gt_vis_pts, axis=1)

    accuracies = []
    for thresh in thresholds:
        acc = (errors < thresh).mean()
        accuracies.append(acc)

    delta_avg = np.mean(accuracies)
    return {
        "delta_avg_vis": delta_avg,
        "per_threshold": dict(zip([1, 2, 4, 8, 16], accuracies)),
        "ATE": errors.mean(),
        "median_error": np.median(errors),
        "num_visible": visible_mask.sum(),
        "errors": errors
    }

pred_last = pred_tracks[0, -1].cpu().numpy()
gt_last = np.array(fL_pts, dtype=np.float32)
pred_vis_last = pred_vis[0, -1].cpu().numpy()

results = tapvid_delta_avg(pred_last, gt_last, pred_vis_last, H, W)

print("\n" + "=" * MAX_FRAMES)
print("         TAP-Vid Official Evaluation Metrics")
print("=" * MAX_FRAMES)
print(f"δ_avg^vis (core metric):  {results['delta_avg_vis']*100:.1f}%")
print(f"ATE (average pixel error):    {results['ATE']:.2f} px")
print(f"Median error:              {results['median_error']:.2f} px")
print(f"Visible points:              {results['num_visible']} / {len(f0_pts)}")
print("-" * MAX_FRAMES)
print("Accuracy per threshold:")
for thresh, acc in results["per_threshold"].items():
    bar = "█" * int(acc * 20)
    print(f"  δ<{thresh:>2}px:  {acc*100:5.1f}%  {bar}")
print("=" * MAX_FRAMES)

print("\nPoint-by-point tracking results:")
print(f"{'Point':>4}  {'Pred(x,y)':>16}  {'GT(x,y)':>16}  {'Error':>8}  {'Visible':>6}  {'Status':>4}")
print("-" * 65)
for i, (e, v) in enumerate(zip(
    np.linalg.norm(pred_last - gt_last, axis=1),
    pred_vis_last
)):
    px, py = pred_last[i]
    gx, gy = gt_last[i]
    status = "✅" if e < 10 else "❌"
    vis_str = f"{v:.2f}"
    print(f"Point{i+1}  ({px:6.1f},{py:6.1f})  ({gx:6.1f},{gy:6.1f})  {e:6.2f}px  {vis_str:>6}  {status}")

os.makedirs("./vis_output", exist_ok=True)

vis = Visualizer(save_dir="./vis_output", pad_value=120, linewidth=2, fps=10)
vis.visualize(
    video=video_tensor,
    tracks=pred_tracks,
    visibility=pred_vis,
    filename="dynamic_seq_003_5pts"
)
print("\n Trajectory video → ./vis_output/dynamic_seq_003_5pts.mp4")

last_frame_vis = video_tensor[0, -1].permute(1, 2, 0).cpu().numpy().astype(np.uint8)
last_frame_vis = cv2.cvtColor(last_frame_vis, cv2.COLOR_RGB2BGR)
for i, ((px, py), (gx, gy)) in enumerate(zip(pred_last, gt_last)):
    cv2.circle(last_frame_vis, (int(gx), int(gy)), 6, (0, 255, 0), 2)
    cv2.circle(last_frame_vis, (int(px), int(py)), 4, (0, 0, 255), -1)
    cv2.line(last_frame_vis, (int(gx), int(gy)), (int(px), int(py)), (0, 255, 255), 1)
    cv2.putText(last_frame_vis, f"P{i+1}", (int(px) + 6, int(py) - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

cv2.imwrite("./vis_output/last_frame_eval.png", last_frame_vis)
print("Comparison image → ./vis_output/last_frame_eval.png")

/Users/cx/Desktop/EndoTrack
读取到 30 帧, 分辨率: (480, 480, 3)

         TAP-Vid 官方评估指标
δ_avg^vis (核心指标):  44.0%
ATE (平均像素误差):    16.22 px
中位误差:              12.21 px
可见点数:              5 / 5
------------------------------
各阈值精度:
  δ< 1px:    0.0%  
  δ< 2px:   20.0%  ████
  δ< 4px:   20.0%  ████
  δ< 8px:   80.0%  ████████████████
  δ<16px:  100.0%  ████████████████████

逐点追踪结果:
   点           预测(x,y)           GT(x,y)        误差      可见    状态
-----------------------------------------------------------------
点1  ( 188.1, 177.2)  ( 201.0, 193.0)   20.36px    1.00  ❌
点2  ( 237.0, 123.2)  ( 238.0, 126.0)    3.01px    1.00  ✅
点3  ( 212.6, 241.6)  ( 219.0, 252.0)   12.21px    1.00  ❌
点4  ( 319.8, 139.6)  ( 293.0, 118.0)   34.40px    1.00  ❌
点5  ( 320.7, 236.1)  ( 321.0, 225.0)   11.14px    1.00  ❌
Video saved to ./vis_output/dynamic_seq_003_5pts.mp4

 轨迹视频 → ./vis_output/dynamic_seq_003_5pts.mp4
对比图   → ./vis_output/last_frame_eval.png


: 